Проверка устойчивости

In [2]:
import pandas as pd

modelv1 = pd.read_excel("../../data/processed/v1_clean_base.xlsx")

In [7]:
import pandas as pd
import statsmodels.formula.api as smf

cols = ["salary", "gender", "experience", "education_type", "region_code", "industry_code"]
data = modelv1[cols].dropna().copy()

selected_industries = ["DeskWork", "Education", "Industry"]
data_sub = data[data["industry_code"].isin(selected_industries)].copy()

min_n = 20
results_sub = []

for (region, industry), g in data_sub.groupby(["region_code", "industry_code"]):
    if len(g) < min_n:
        continue

    if g["gender"].nunique() < 2:
        continue

    model = smf.ols(
        'salary ~ C(gender, Treatment(reference="Женский")) + experience + C(education_type)',
        data=g
    ).fit()

    for name, coef in model.params.items():
        if "gender" in name:
            results_sub.append({
                "region_code": region,
                "industry_code": industry,
                "n": len(g),
                "term": name,
                "coef": coef,
                "pvalue": model.pvalues[name],
                "r2": model.rsquared
            })

robust_by_industry = pd.DataFrame(results_sub).sort_values(["region_code", "industry_code"])
print(robust_by_industry.head(20))

robust_by_industry.to_excel("../../data/processed/robust_by_industry.xlsx", index=False)

    region_code industry_code    n  \
0             2      DeskWork   83   
1             2     Education   34   
2             2      Industry   21   
3             5      DeskWork   46   
4             5      Industry   31   
5            10      DeskWork   20   
6            11      DeskWork   42   
7            13      DeskWork   37   
8            16      DeskWork   98   
9            16     Education   42   
10           16      Industry   48   
11           18      DeskWork   43   
12           18      Industry   27   
13           19      DeskWork   34   
14           21      DeskWork   88   
15           21      Industry   22   
16           22      DeskWork   59   
17           22      Industry   24   
18           23      DeskWork  154   
19           23     Education   62   

                                                 term          coef  \
0   C(gender, Treatment(reference="Женский"))[T.Му...   7648.037901   
1   C(gender, Treatment(reference="Женский"))[T.Му...  -478